In [2]:
from typing import List, Tuple
from helper_class import MixtureMeta
import pandas as pd
from pathlib import Path
import os
import librosa
import numpy as np
from faster_whisper import WhisperModel

/home/jamin/mambaforge/envs/fw311/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
def load_mixture_meta(path: Path) -> pd.DataFrame:
    return pd.read_csv(path)

In [4]:
metas = load_mixture_meta(Path(os.pardir, Path("Output", "manifest.csv")))

In [ ]:
# extract row whose transcript contains ("any speaker", None) or ("any speaker", "None") or ("any speaker", np.nan) where transcript is a list of tuples (speaker, transcript)
row_none = metas[metas["transcript"].apply(lambda x: any((t[0] == "any speaker" and (t[1] is None or t[1] == "None" or (isinstance(t[1], float) and np.isnan(t[1]))) for t in eval(x))))]


In [7]:
row_none

,clip_id,audio_path,transcript,overlap_ratio_target,overlap_ratio_actual,max_speakers,snr_db,noise_type,overlap_mask_path,source_files,noise_files


In [2]:
import os
print(os.environ.get("LD_LIBRARY_PATH"))

/home/jamin/mambaforge/envs/fw311/lib/python3.11/site-packages/nvidia/cublas/lib:/home/jamin/mambaforge/envs/fw311/lib/python3.11/site-packages/nvidia/cudnn/lib:


In [3]:
import ctypes
ctypes.CDLL("libcublas.so.12")
ctypes.CDLL("libcudnn.so.9")
print("CUDA libs load OK")

CUDA libs load OK


In [4]:
def load_mixture_meta(path: Path) -> pd.DataFrame:
    return pd.read_csv(path)
def load_mixture_audio(path: Path, sr: int = 16000) -> Tuple[np.ndarray, int]:
    audio, sr = librosa.load(path, sr=sr)
    return audio, sr

In [5]:
meta_df = load_mixture_meta(Path(os.pardir,"Output", "manifest.csv"))

In [6]:
meta_df.clip_id.filter

<bound method NDFrame.filter of 0       mix_0.00_2_None_D_0000000
1       mix_0.00_2_None_P_0000001
2       mix_0.00_2_None_T_0000002
3        mix_0.00_2_7.4_D_0000003
4        mix_0.00_2_7.4_P_0000004
                  ...            
3715       mix_0.40_2_0_P_0003715
3716       mix_0.40_2_0_T_0003716
3717      mix_0.40_2_-5_D_0003717
3718      mix_0.40_2_-5_P_0003718
3719      mix_0.40_2_-5_T_0003719
Name: clip_id, Length: 3720, dtype: str>

In [7]:
# filter out clip ids that starts with mix_0.14
test_meta = meta_df[meta_df.clip_id.str.startswith("mix_0.14_2_None_D_0000144")].head(1)
test_meta

,clip_id,audio_path,transcript,overlap_ratio_target,overlap_ratio_actual,max_speakers,snr_db,noise_type,overlap_mask_path,source_files,noise_files
144,mix_0.14_2_None_D_0000144,Output/audio/mix_0.14_2_None_D_0000144.wav,"[('125-121342-0038', ""THEY DON'T THEY JUST WAN...",0.14,0.14,2,NaN,D,Output/masks/mix_0.14_2_None_D_0000144.npy,[PosixPath('/home/jamin/Year3Proj/Data/LibriSp...,[]


In [8]:
print(test_meta.audio_path.values[0])
print(Path(os.pardir, test_meta.audio_path.values[0]))

Output/audio/mix_0.14_2_None_D_0000144.wav
../Output/audio/mix_0.14_2_None_D_0000144.wav


In [9]:
# model = WhisperModel("large-v3", device="cuda")

# segments, info = model.transcribe(Path(os.pardir, test_meta.audio_path.values[0]))
# for segment in segments:
#     print("[%.2fs -> %.2fs] %s" % (segment.start, segment.end, segment.text))


In [10]:
# import torch
# from transformers import pipeline

# device = 0 if torch.cuda.is_available() else -1

# asr = pipeline(
#     "automatic-speech-recognition",
#     model="facebook/wav2vec2-large-960h",
#     device=0,
# )
# audio = load_mixture_audio(Path(os.pardir, test_meta.audio_path.values[0]))[0]
# result = asr(audio)
# print(result["text"])

In [11]:
# from transformers import Wav2Vec2Processor, Wav2Vec2ForCTC
# import torch

# # load model and processor
# processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-large-960h")
# model = Wav2Vec2ForCTC.from_pretrained("facebook/wav2vec2-large-960h")
    
# # load dummy dataset and read soundfiles
# audio = load_mixture_audio(Path(os.pardir, test_meta.audio_path.values[0]))[0]

# # tokenize
# input_values = processor(audio, return_tensors="pt", padding="longest").input_values  # Batch size 1

# # retrieve logits
# logits = model(input_values).logits

# # take argmax and decode
# predicted_ids = torch.argmax(logits, dim=-1)
# transcription = processor.batch_decode(predicted_ids)


In [12]:
# print(test_meta.transcript.values[0].astype(str))

# print(" ".join(seg[1] for seg in test_meta.transcript.values[0]))

In [13]:
# import nemo.collections.asr as nemo_asr
# asr_model = nemo_asr.models.ASRModel.from_pretrained(model_name="nvidia/parakeet-tdt-0.6b-v3")
# output = asr_model.transcribe([Path(os.pardir, test_meta.audio_path.values[0])])
# print(output[0].text)


In [14]:
# import whisperx
# device = "cuda"   # or "cpu"
# batch_size = 1
# compute_type = "int8"
# model = whisperx.load_model("large-v2", device, compute_type=compute_type)
# audio = whisperx.load_audio(Path(os.pardir) / test_meta.audio_path.values[0])

# result = model.transcribe(audio, batch_size=batch_size)

In [ ]:
# import nemo.collections.asr as nemo_asr
# asr_model = nemo_asr.models.ASRModel.from_pretrained("nvidia/parakeet-tdt-0.6b-v2")
# transcript = asr_model.transcribe([test_meta.audio_path.values[0]])

[NeMo W 2026-04-04 00:15:05 megatron_init:62] Megatron num_microbatches_calculator not found, using Apex version.
OneLogger: Setting error_handling_strategy to DISABLE_QUIETLY_AND_REPORT_METRIC_ERROR for rank (rank=0) with OneLogger disabled. To override: explicitly set error_handling_strategy parameter.
No exporters were provided. This means that no telemetry data will be collected.
Cancellation requested; stopping current tasks.


KeyboardInterrupt: 

In [24]:
import pickle
with open("funasr_output.pkl", "rb") as f:
    funasr_output = pickle.load(f)


In [29]:
funasr_output[0][0]

{'key': 'rand_key_1t9EwL56nGisi',
 'text': "They don't. They just want the fun of eating it all over again. The matron doesn't want to repeat her own feelings, but she's just a part of the world. Again, Ventnor's rifle cracked. One of the colors of the world is down, another's down, more did he reproach himself for feelings that were bound to the horseman, but his sworn foot soldiers, a forest of shining planks above them, clearly to us came their battle cries. Again, Ventnor's rifle cracked. I detest poor people. I hate them for being poor. Poverty may have been beautiful for the horseman, but it's one foot in a forest and truly a big faint radiance. It was her eyes, her great wide eyes whose clear form magnified us. Over us whined the bullets from the covering guns. The fathom deeper by lie, with old desires restrained before. No, I heard a song. The matron doesn't want to repeat her. We just want to repeat her honeymoon. How the smell of it suddenly cleared the air and threw everyon